# Fast Transformer Decoding: One Write-Head is All You Need

## 摘要
>原因在于反复加载庞大的"键"和"值"张量所带来的内存带宽开销。我们提出一种称为多查询注意力（multi-query attention）的变体，其中键和值在所有不同的注意力"头"之间共享，从而大幅缩减这些张量的大小，进而降低增量解码的内存带宽需求。实验验证表明，由此得到的模型解码速度确实大幅提升，且相比基线模型仅有轻微的质量下降。

## 背景

### 点积注意力机制：查询q和m对(k,v)键值对，输出y
$$
y = softmax(Q*K^{T})*V
$$

仅为一个单维的点积注意力
<p  align="center">
    <img src="./static/dotProduct.png">
</p>
其中，einsum是万能张量运算工具，einsum("输入1维度,输入2维度->输出维度", 张量1, 张量2)

In [ ]:
import torch

def dot_attention(q,k,v,mask=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn, v)
    return output, attn

### 多头注意力
> 并行训练 h 组独立的 Q/K/V，模型能同时从 h 个不同角度提取信息，点积注意力处理维度：
head_dim = embedding_dim / num_head
<p align="center">
    <img src="./static/multiHead.png">
</p>

In [ ]:
import torch.nn as nn

def multiHeadAttention(q, k, v, num_heads, embedding_dim, mask=None):
    batch_size = q.size(0)
    d_k = embedding_dim // num_heads
    W_q = nn.Linear(embedding_dim, embedding_dim)
    W_k = nn.Linear(embedding_dim, embedding_dim)
    W_v = nn.Linear(embedding_dim, embedding_dim)
    W_o = nn.Linear(embedding_dim, embedding_dim)

    # 线性投影生成 Q/K/V
    q = W_q(q)
    k = W_k(k)
    v = W_v(v)

    # 切分维度 (batch_size, num_heads, seq_len, d_k) 
    q = q.view(batch_size, -1, num_heads, d_k).transpose(1, 2)
    k = k.view(batch_size, -1, num_heads, d_k).transpose(1, 2)
    v = v.view(batch_size, -1, num_heads, d_k).transpose(1, 2)

    attn_output, _ = dot_attention(q, k, v, mask)

    # (batch_size, seq_len, num_heads*d_k) → (batch_size, seq_len, embedding_dim)
    attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, embedding_dim)

    output = W_o(attn_output)
    return output

### 批量多头性能分析
b：batch size（批量大小，一次喂多少条数据）

n：序列长度（一句话有多少词，自注意力中m=n）

d：模型总维度（embedding_dim，比如 512/1024）

h：注意力头数

k=v=d/h：单个头的维度（论文标准设置）

假设：源序列长度 = 目标序列长度(m = n)

**总算术运算量**（加减乘除总次数）为 $\Theta (bnd^2)$,一次矩阵乘法的运算量是abc,如 A(a,b)@B(b,c) 

**总内存访问量** $O(bnd + bhn^2 + d^2)$。一次矩阵读/写次数 = abc,如A.shape = (a,b,c)

bnd：输入词向量、Q/K/V、输出特征（主要数据）；

$bhn^2$：注意力权重矩阵（多头的分数表）；

$d^2$：4 个投影矩阵 $W_q/W_k/W_v/W_o$（模型参数，很小）

核心比值：内存访问 ÷ 算术运算 = $O(1/k + 1/bn)$

比值越小 → 算得多、读得少 → GPU 利用率拉满；

比值越大 → 算得少、读得多 → GPU 一直在等数据，性能极差。

### 增量多头注意力
> 后面的词依赖前面的所有词 → 这就是数据依赖，不能并行算所有词。


训练	并行批量计算	提前知道完整句子，所有词同时算注意力

生成	增量逐个计算	只能一个词一个词生成，不知道未来的词

如果生成时每次都重新计算所有词的 K/V，速度会极慢；

代码的解决方案：缓存之前所有词的 K 和 V，只计算当前新词的 K/V，拼接到缓存里 → 不用重复计算

In [ ]:
import torch
import torch.nn.functional as F
def MultiheadSelfAttentionIncremental(x,prev_K,prev_V,W_q,W_k,W_v,W_o):
    """
    增量式多头自注意力
    """
    batch_size, d_model = x.shape
    num_heads = W_q.out_features // (prev_K.shape[-1])
    d_k = prev_K.shape[-1]

    q = W_q(x)  # [b, d_model]
    k = W_k(x)  # [b, d_model]
    v = W_v(x)  # [b, d_model]

    q = q.view(batch_size, 1, num_heads, d_k).transpose(1, 2)
    k = k.view(batch_size, 1, num_heads, d_k).transpose(1, 2)
    v = v.view(batch_size, 1, num_heads, d_k).transpose(1, 2)


    new_K = torch.cat([prev_K, k], dim=2)  # [b, h, m+1, d_k]
    new_V = torch.cat([prev_V, v], dim=2)  # [b, h, m+1, d_k]


    scores = torch.matmul(q, new_K.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    attn_weights = F.softmax(scores, dim=-1)  # 注意力权重 [b, h, 1, m+1]

    # 加权求和 V
    attn_output = torch.matmul(attn_weights, new_V)  # [b, h, 1, d_k]

    # [b, h, 1, dk] → [b, 1, h, dk] → [b, 1, d_model]
    attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, 1, d_model)
    y = W_o(attn_output).squeeze(1)  # 输出 [b, d_model]

    return y, new_K, new_V

### 增量注意力性能分析

b：batch 大小；

n：总共生成n个 token，增量函数循环调用n次；

d：模型隐层维度；h：头数；k=d/h

总算术运算总量：$\Theta(bnd^2)$

总内存访问开销：$\Theta(bn^2d+nd^2)$

第一项$bn^2d$：由每一步都要全量读取KV 缓存带来；

第二项$nd^2$：由 4 个投影参数$W_q,W_k,W_v,W_o$重复读取带来；

访存 / 计算比：$\Theta\left(\dfrac{n}{d}+\dfrac1b\right)$

当生成场景满足 $n\approx d$（上下文很长）或 $b\approx1$（日常单条文本生成）时，比值趋近于 1，显存带宽成为推理瓶颈（GPU 算力闲置、一直在等数据）。

优化$\boldsymbol{\dfrac1b}$很简单：在显存允许前提下，增大 batch 批量；

优化$\boldsymbol {\dfrac{n}{d}}$难度更高：该项来自每一步推理需要完整加载全部历史 KV 张量（KV 体积随序列变长线性膨胀）。传统解法两种：① 限制最大上下文长度n；② 局部注意力 / 历史 KV 压缩，让每个 token 只关注少量历史位置。

本文提出全新正交优化思路：查询Q维持多头维度，把 K、V 的多头 (h) 维度直接抹除，从张量尺寸上压缩 KV 缓存。

原版$\mathrm{K/V}:[b,h,n,k]$，带h个头；

新方案：$\boldsymbol{Q}$保留多头h，$\boldsymbol{K/V}$去掉头维度，$[b,n,d]$，直接砍掉 KV 的h倍体积，大幅降低$bn^2d$项，从源头压小$\frac nd$，和传统局部注意力互不冲突（正交方案）。


## 多查询注意力

多头注意力（MHA）：并行的 h 个注意力头，每个头都有独立的 Q、K、V、输出线性层；

多查询注意力（MQA）：和多头注意力几乎完全一样，唯一区别：所有注意力头共享同一组 K 和 V。

**批量版本：** K = tf.einsum("bmd,dk->bmk", M, P_k) — W_k 形状从 [h,d,k] 降为 [d,k]，K 形状从 [b,h,m,k] 降为 [b,m,k]

**增量版本：** prev_K 形状从 [b,h,m,k] 降为 [b,m,k]，prev_V 从 [b,h,m,v] 降为 [b,m,v]

In [ ]:
import torch
import torch.nn.functional as F

def MultiqueryAttentionBatched(X ,M ,mask ,W_q ,W_k ,W_v ,W_o):
    """
    Multi-Query Attention (MQA)
    Q 多头 / K,V 共享无头维度
    输入：
        X: [batch_size, seq_len_q, d_model]
        M: [batch_size, seq_len_k, d_model]
        mask: [batch_size, num_heads, seq_len_q, seq_len_k]
    输出：
        Y: [batch_size, seq_len_q, d_model]
    """
    batch_size = X.shape[0]
    d_model = X.shape[-1]
    num_heads = W_q.out_features // (W_q.in_features // 8)  
    d_k = W_q.out_features // num_heads

    # q保留多头，k,v共享无头维度
    q = W_q(X)  # [b, n_q, d_model]
    q = q.view(batch_size, -1, num_heads, d_k).transpose(1, 2)  # [b, h, n_q, d_k]


    k = W_k(M)  # [b, n_k, d_k]
    v = W_v(M)  # [b, n_k, d_k]

    k = k.unsqueeze(1)  # [b, 1, n_k, d_k] → 广播匹配所有头
    
    scores = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    scores = scores + mask  # 加入掩码
    attn_weights = F.softmax(scores, dim=-1)  # [b, h, n_q, n_k]

    v = v.unsqueeze(1)  # [b, 1, n_k, d_k]
    attn_output = torch.matmul(attn_weights, v)  # [b, h, n_q, d_k]

    attn_output = attn_output.transpose(1, 2).contiguous()  # [b, n_q, h, d_k]
    attn_output = attn_output.view(batch_size, -1, d_model)  # [b, n_q, d_model]
    y = W_o(attn_output)

    return y

# 增量
def MultiquerySelfAttentionIncremental(
    x,
    prev_K,  # MQA核心：无头维度 [b, m, d_k]
    prev_V,  
    W_q,
    W_k,
    W_v,
    W_o
):
    """
    输入：
        x:      [batch_size, d_model] → 当前单个新词
        prev_K: [batch_size, m, d_k]  → 历史K缓存（无头）
        prev_V: [batch_size, m, d_k]  → 历史V缓存（无头）
    输出：
        y:      [batch_size, d_model]
        new_K:  [batch_size, m+1, d_k]
        new_V:  [batch_size, m+1, d_k]
    """
    batch_size, d_model = x.shape
    num_heads = W_q.out_features // prev_K.shape[-1]
    d_k = prev_K.shape[-1]

    q = W_q(x)  # [b, d_model]
    q = q.view(batch_size, 1, num_heads, d_k).transpose(1, 2)  # [b, h, 1, d_k]

    k = W_k(x)  # [b, d_k]
    v = W_v(x)  # [b, d_k]
    k = k.unsqueeze(1)  # [b, 1, d_k]
    v = v.unsqueeze(1)  # [b, 1, d_k]

    new_K = torch.cat([prev_K, k], dim=1)  # [b, m+1, d_k]
    new_V = torch.cat([prev_V, v], dim=1)  # [b, m+1, d_k]

    k_expand = new_K.unsqueeze(1)  # [b, 1, m+1, d_k] → 广播匹配多头Q
    scores = torch.matmul(q, k_expand.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    attn_weights = F.softmax(scores, dim=-1)  # [b, h, 1, m+1]

    v_expand = new_V.unsqueeze(1)  # [b, 1, m+1, d_k]
    attn_output = torch.matmul(attn_weights, v_expand)  # [b, h, 1, d_k]

    attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, 1, d_model)
    y = W_o(attn_output).squeeze(1)  # [b, d_model]

    return y, new_K, new_V

MHA 为什么 KV 体积大？—— 同一个 token 重复生成 h 份 K/V MHA：

$W_k\in[h,d,k]$：h 套完全独立的 K 投影权重

同一个输入 token 向量，经过 h 个不同的 $W_k$，算出 h 份独立 K 特征

K 张量：$\boldsymbol{[b,h,n,k]}$

MQA 的 K = 直接学习完整的单头特征 可学习

MQA让网络直接学习一个完整 K/V：包含了所有需要的语义信息；

### MQA性能分析

MQA 的总算术运算量 和标准多头注意力（MHA），都是 $\Theta(bnd^2)$；

连续生成 n 个词，MQA 的总显存访问量 变成了 $\Theta(bnd + bn^2k + nd^2)$

访存计算比：$\Theta\left(\frac{1}{d} + \frac{n}{dh} + \frac{1}{b}\right)$。最慢速度的 $\frac{n}{d}$ 项，缩小了 h 倍


In [ ]:
## 测试对比
